# PINNs for Mechanical Engineers — Tutorial 4: Natural Convection in a Buoyant Cavity
### Multiphysics at last: temperature drives the flow, and the flow moves the heat

*📺 Video on YouTube | 🐍 Python · PyTorch | Advanced*

*Notebook repository: [github.com/rraj-io/PINN_for_Mechanical_Engineer](https://github.com/rraj-io/PINN_for_Mechanical_Engineer)*

---

## Series Roadmap

This series applies physics-informed neural networks to the problems mechanical engineers actually deal with — fluids, heat, solids, and the messy places where they couple. Tutorials 1–3 solved a flow, heat, and a stressed solid **in isolation**. This one is where the messiness starts: two physics that **feed back on each other**. Temperature makes the fluid buoyant, so it rises; the moving fluid carries heat around, which changes the temperature. Neither field can be solved without the other.

| # | Tutorial | Focus | Status |
|---|----------|-------|--------|
| 1 | 2D Navier–Stokes — lid-driven cavity | coupled, nonlinear flow | ✅ Published |
| 2 | 2D Steady Heat Conduction | mixed Dirichlet/Neumann BCs | ✅ Published |
| 3 | Linear Elasticity — plate with a hole | vector fields, stress concentration | ✅ Published |
| **4** | **Natural Convection — buoyant cavity** | **multiphysics coupling** | **📍 You are here** |
| 5 | Euler–Bernoulli Beam | 4th-order derivatives | Planned |
| 6 | Backward-Facing Step | flow separation & reattachment | Planned |
| 7 | Unsteady Vortex Shedding — cylinder wake | the hard one: unsteady + failure modes | Planned |

Every tutorial is standalone and validated against a reference solution. This one leans on a recent, peer-reviewed **PINN benchmark** that solved exactly this problem — Fowler, McDevitt & Roy, *Scientific Reports* **14**, 15203 (2024) — whose formulation we follow and whose results (validated against the classic de Vahl Davis 1983 CFD benchmark) we compare against.

> **New to PINNs?** This series assumes you're already comfortable with the
> fundamentals: autograd, collocation points, the residual loss, the
> Adam → L-BFGS recipe. If you're starting from zero — or you just want the
> deeper, build-it-from-scratch treatment in one place — I wrote a 100+ page
> ebook that does exactly that: from your very first autograd call through 1D
> ODEs, the Poisson equation, and nonlinear Burgers. It's the foundation this
> series builds on.
>
> 👉 **[Get *PINNs from Scratch* on Gumroad](https://rajrohit403.gumroad.com/l/pinn_tutorial)**

---


## 1 — What You Will Learn

By the end of this notebook you will have:

- Solved the **coupled Boussinesq natural-convection problem** with a PINN, using the **stream-function/vorticity formulation** — the choice that makes it actually converge

- Understood why the naive **primitive-variable** approach $(u,v,p,T)$ gets **trapped in the pure-conduction solution**, and why $(\psi, T)$ escapes it

- Predicted just **two fields** $\psi$ and $T$, and recovered velocity, vorticity, and heat flux by autograd — with **continuity satisfied exactly** and **pressure eliminated entirely**
- Baked the boundary conditions in as **hard constraints**: no-slip and the hot/cold wall temperatures hold *exactly, by construction*, leaving only the insulated walls as a soft loss term
- Assembled the residuals from the **vorticity transport** and **energy** equations, taking derivatives of $\psi$ up to **fourth order**
- Validated against the **de Vahl Davis (1983)** benchmark (Nusselt number, peak velocities) as reproduced by the PINN study above
- Turned the **Rayleigh dial** and seen how convergence gets harder as convection strengthens — the honest on-ramp to the turbulence question

**Prerequisites:** the PINN fundamentals, plus all three prior tutorials — and this time you use them together. Navier–Stokes machinery is Tutorial 1's; the energy equation and the insulated Neumann wall are Tutorial 2's; the multi-output network is Tutorial 3's. What's new is that two physics are **wired together** and must hold at once.

⏱️ **Runtime:** the heaviest solve in the series. The vorticity residual needs 4th-order autograd, and natural convection takes many L-BFGS iterations to develop. Use a GPU. Expect this to want real training time, more than any earlier tutorial.


## 2 — The problem and its reference

Here the series changes character. Tutorials 2 and 3 had *exact* analytical answers. Natural convection is **nonlinear and coupled** — no closed form exists. We validate against a **published benchmark**: the classic de Vahl Davis (1983) CFD solution, and the recent PINN reproduction of it by Fowler et al. (2024), whose formulation we adopt here.

### The physics — and why the formulation matters

The obvious approach is to predict $(u, v, p, T)$ and enforce continuity, two momentum equations, and energy. It **fails**: the network settles into the trivial *pure-conduction* solution (linear temperature, zero flow) and never develops convection. The buoyancy term $Ra\,Pr\,T \approx 7100\,T$ creates a huge pressure that an ordinary network can't represent, and the zero-flow state is a deep local minimum. (Fowler et al. report the same trap; a direct run confirms it — the flow stays *exactly* zero.)

The fix is the **stream-function/vorticity formulation**. We predict only two fields — the stream function $\psi$ and temperature $T$ — and everything else follows:

$$u=\frac{\partial\psi}{\partial y},\qquad v=-\frac{\partial\psi}{\partial x},\qquad \zeta=-\nabla^2\psi\ \ (\text{vorticity}).$$

Defining velocity this way makes **continuity exact for free** ($\partial u/\partial x+\partial v/\partial y\equiv 0$), and taking the curl of the momentum equations **eliminates pressure entirely**. Two coupled equations remain:

$$
\begin{aligned}
\text{vorticity transport:}\quad & u\,\zeta_x + v\,\zeta_y = Pr\,\nabla^2\zeta + Ra\,Pr\,\frac{\partial T}{\partial x}\\[4pt]
\text{energy:}\quad & u\,T_x + v\,T_y = \nabla^2 T
\end{aligned}
$$

Buoyancy now enters as $Ra\,Pr\,\partial T/\partial x$ — a horizontal temperature gradient generating vorticity. The price is that $\nabla^2\zeta=\nabla^4\psi$ needs **fourth-order** derivatives of $\psi$; the reward is a problem that actually converges.

### Dimensionless numbers

$$Ra=\frac{g\beta\,\Delta T\,L^3}{\nu\alpha}\ \ (\text{Rayleigh}),\qquad Pr=\frac{\nu}{\alpha}=0.71\ \ (\text{air}).$$

$Pr=0.71$ throughout (to match the benchmark); $Ra$ is the dial. Main run: $Ra=10^4$.

### Boundary conditions — the differentially heated cavity

Unit square, hot left wall, cold right wall, insulated top/bottom, **no-slip everywhere**:

| Wall | Location | Thermal | Velocity |
|------|----------|---------|----------|
| Left (hot) | $x=0$ | $T=1$ | no-slip |
| Right (cold) | $x=1$ | $T=0$ | no-slip |
| Top / Bottom | $y=1,\,0$ | $\partial T/\partial y=0$ | no-slip |

Following Fowler et al., no-slip and the hot/cold temperatures are enforced as **hard constraints** (Section 6); only the insulated walls are a soft loss term.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Arc

BG="#f7f6f2"; HOT="#E76F51"; COLD="#3B6FB0"; INK="#1A1A2E"; TEAL="#2A9D8F"
fig, ax = plt.subplots(figsize=(6.2, 6.2)); fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
ax.plot([0,0],[0,1], color=HOT,  lw=6, solid_capstyle='butt')
ax.plot([1,1],[0,1], color=COLD, lw=6, solid_capstyle='butt')
ax.plot([0,1],[1,1], color=INK,  lw=3, solid_capstyle='butt')
ax.plot([0,1],[0,0], color=INK,  lw=3, solid_capstyle='butt')
for xx in np.linspace(0.03, 0.97, 16):
    ax.plot([xx, xx+0.03], [1.0, 1.05], color=INK, lw=0.8)
    ax.plot([xx, xx+0.03], [-0.05, 0.0], color=INK, lw=0.8)
ax.text(-0.10,0.5,"hot wall\n$T=1$\nno-slip",color=HOT,ha='center',va='center',fontsize=10,rotation=90,weight='bold')
ax.text(1.10,0.5,"cold wall\n$T=0$\nno-slip",color=COLD,ha='center',va='center',fontsize=10,rotation=90,weight='bold')
ax.text(0.5,1.13,r"insulated: $\partial T/\partial y=0$,  no-slip",color=INK,ha='center',fontsize=9.5)
ax.text(0.5,-0.13,r"insulated: $\partial T/\partial y=0$,  no-slip",color=INK,ha='center',fontsize=9.5)
ax.add_patch(Arc((0.5,0.5),0.62,0.62,angle=0,theta1=20,theta2=320,color=TEAL,lw=2.2))
th=np.deg2rad(20)
ax.add_patch(FancyArrowPatch((0.5+0.31*np.cos(th),0.5+0.31*np.sin(th)),
    (0.5+0.31*np.cos(th)-0.001,0.5+0.31*np.sin(th)-0.03),arrowstyle='-|>',mutation_scale=18,color=TEAL))
ax.text(0.5,0.5,"buoyant\ncirculation",color=TEAL,ha='center',va='center',fontsize=9,style='italic')
ax.add_patch(FancyArrowPatch((1.28,0.9),(1.28,0.6),arrowstyle='-|>',mutation_scale=18,color=INK))
ax.text(1.34,0.75,"$g$",color=INK,fontsize=12)
ax.add_patch(FancyArrowPatch((-0.02,-0.02),(0.22,-0.02),arrowstyle='-|>',mutation_scale=12,color=INK))
ax.add_patch(FancyArrowPatch((-0.02,-0.02),(-0.02,0.22),arrowstyle='-|>',mutation_scale=12,color=INK))
ax.text(0.23,-0.055,"$x$",fontsize=11); ax.text(-0.075,0.23,"$y$",fontsize=11)
ax.set_xlim(-0.35,1.5); ax.set_ylim(-0.25,1.25); ax.set_aspect('equal'); ax.axis('off')
ax.set_title("Differentially heated square cavity", color=INK, fontsize=12, pad=10)
plt.tight_layout(); plt.show()

## 3 — Imports and Setup

Same three libraries as every tutorial in the series — `torch`, `numpy`, `matplotlib` — a seed, and automatic GPU selection. The constants block holds the two dimensionless numbers from Section 2. Because the equations are already **non-dimensional**, every field is $O(1)$ by construction: $T\in[0,1]$, and the velocities scale with $Ra$ but stay well-behaved at the values we use. That's the same "keep everything $O(1)$ so training is stable" discipline as Tutorial 3's non-dimensionalization — here it comes for free from the Boussinesq scaling.

`RA` is the dial. We set it to $10^4$ for the main run; changing this single number is the whole "watch the PINN strain" experiment later — crank it to $10^5$ and the boundary layers thin until the network struggles.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math, time

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

RA = 1.0e4        # Rayleigh number  -> the dial
PR = 0.71         # Prandtl number   -> air; fixed to match the benchmark
RA_PR = RA * PR   # buoyancy coefficient, reused in the vorticity residual
PI = math.pi
print(f"Ra = {RA:.0e}, Pr = {PR}")

## 4 — The reference: benchmark values + diagnostics

No analytical field exists, so the reference is a table of scalars plus the functions that extract the same scalars from our PINN. Values are the de Vahl Davis (1983) benchmark for $Pr=0.71$ (as tabulated and reproduced by Fowler et al. 2024): the average Nusselt number on the hot wall, the peak horizontal velocity on the vertical mid-plane, and the peak vertical velocity on the horizontal mid-plane.

The diagnostic functions reference `model` and `fields`, defined in Sections 5–6; Python resolves those names at call time, so defining the diagnostics here is fine.

In [ ]:
# de Vahl Davis (1983), Pr = 0.71 (as reproduced in Fowler, McDevitt & Roy 2024)
BENCHMARK = {
    1e3: dict(Nu=1.118, u_max=3.649,  v_max=3.697),
    1e4: dict(Nu=2.243, u_max=16.178, v_max=19.617),
    1e5: dict(Nu=4.519, u_max=34.730, v_max=68.590),
}

def nusselt_hot_wall(n=200):
    "Average Nusselt number on the hot wall (x=0): mean of -dT/dx."
    y = torch.linspace(0, 1, n, device=device).reshape(-1, 1)
    x = torch.zeros_like(y).requires_grad_(True)
    _, T = fields(x, y)
    T_x = torch.autograd.grad(T, x, torch.ones_like(T))[0]
    return (-T_x).mean().item()

def peak_velocities(n=200):
    "u_max on the vertical mid-plane (x=0.5); v_max on the horizontal mid-plane (y=0.5)."
    y = torch.linspace(0, 1, n, device=device).reshape(-1, 1).requires_grad_(True)
    x = torch.full_like(y, 0.5)
    psi, _ = fields(x, y)
    u = torch.autograd.grad(psi, y, torch.ones_like(psi))[0]     # u = dpsi/dy
    u_max = u.abs().max().item()
    x2 = torch.linspace(0, 1, n, device=device).reshape(-1, 1).requires_grad_(True)
    y2 = torch.full_like(x2, 0.5)
    psi2, _ = fields(x2, y2)
    v = -torch.autograd.grad(psi2, x2, torch.ones_like(psi2))[0] # v = -dpsi/dx
    v_max = v.abs().max().item()
    return u_max, v_max

print(f"Benchmark target @ Ra = {RA:.0e}: {BENCHMARK[RA]}")

## 5 — Collocation and boundary points

Because no-slip and the hot/cold temperatures are hard-constrained, we need collocation points only for (a) the **interior** residuals and (b) the **insulated top/bottom walls** (the one soft BC). Following Fowler et al., points are uniform random; we use fewer than their 10,000 to keep a single step affordable — increase `N_f` for production accuracy.

In [ ]:
gen = torch.Generator().manual_seed(42)
def rc(n): return torch.rand(n, 1, generator=gen)

N_f = 4000                                   # interior collocation points
x_f = rc(N_f).to(device).requires_grad_(True)
y_f = rc(N_f).to(device).requires_grad_(True)

N_w = 200                                    # insulated-wall points (top & bottom)
x_bot = rc(N_w).to(device); y_bot = torch.zeros(N_w, 1, device=device).requires_grad_(True)
x_top = rc(N_w).to(device); y_top = torch.ones (N_w, 1, device=device).requires_grad_(True)

print(f"interior: {N_f}  |  insulated-wall points: {2*N_w}")

In [ ]:
def np2(t):
    return t.detach().cpu().numpy()

HOT="#E76F51"; COLD="#3B6FB0"; INK="#1A1A2E"; TEAL="#2A9D8F"; BG="#f7f6f2"
plt.figure(figsize=(6.0, 6.0)); plt.gca().set_facecolor(BG); plt.gcf().patch.set_facecolor(BG)

# interior collocation points (for the residuals)
plt.scatter(np2(x_f), np2(y_f), s=3, alpha=0.28, color=TEAL, label=f"interior collocation ({N_f})")

# insulated top/bottom walls — the only soft BC, so the only walls needing points
plt.scatter(np2(x_bot), np2(y_bot), s=14, color=INK, marker="s", label=f"insulated walls: ∂T/∂y=0 (soft, {2*N_w})")
plt.scatter(np2(x_top), np2(y_top), s=14, color=INK, marker="s")

# hot/cold walls are HARD-constrained (baked into the architecture) -> no collocation points
plt.plot([0,0],[0,1], color=HOT,  lw=6, solid_capstyle='butt')
plt.plot([1,1],[0,1], color=COLD, lw=6, solid_capstyle='butt')
plt.plot([], [], color=HOT,  lw=6, label="hot wall T=1 + no-slip (HARD — no points)")
plt.plot([], [], color=COLD, lw=6, label="cold wall T=0 + no-slip (HARD — no points)")

plt.xlabel("x"); plt.ylabel("y")
plt.title("Where the PINN is asked to satisfy physics")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=1, framealpha=0.95, fontsize=8)
plt.gca().set_aspect("equal"); plt.xlim(-0.03, 1.03); plt.ylim(-0.03, 1.03)
plt.tight_layout(); plt.show()

## 6 — The network

Two outputs: $\psi_0$ and $T_0$, the *raw* network outputs that Section 7 turns into the constrained $\psi$ and $T$. Following Fowler et al., the 2D architecture is **7 hidden layers of 64** with `tanh` (smooth — we need up to 4th derivatives), no output activation, Xavier init.

Everything else is unchanged from Tutorial 3. **`tanh`** activations, because the residuals need *second* derivatives (the viscous and diffusive Laplacians), so the network must be smooth to second order. The `n_hidden` / `n_layers` parametrization is the same.

In [ ]:
class PINN(nn.Module):
    "Input (x, y) -> raw outputs (psi0, T0)."
    def __init__(self, n_hidden=64, n_layers=7):
        super().__init__()
        layers = [nn.Linear(2, n_hidden), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(n_hidden, n_hidden), nn.Tanh()]
        layers += [nn.Linear(n_hidden, 2)]
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    def forward(self, x, y):
        out = self.net(torch.cat([x, y], dim=1))
        return out[:, 0:1], out[:, 1:2]     # psi0, T0

model = PINN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Trainable parameters: {n_params}")

## 7 — Hard constraints (the key to convergence)

Instead of penalizing the walls with loss terms, we **build the boundary conditions into the outputs**, so they hold exactly, for free — and the network only has to learn the interior. This is the single most important trick for making natural convection converge.

**No-slip, all four walls.** No-slip means both velocity components vanish on every wall, i.e. $\psi$ **and** its normal derivative are zero there. Multiply the raw output by the square of a function that vanishes on the boundary:

$$b(x,y)=16\,x(1-x)\,y(1-y),\qquad \psi=b^2\,\psi_0.$$

Because $b=0$ on every wall, $\psi=0$ there; and because it's *squared*, $\nabla\psi=0$ there too — so $u=v=0$ on all walls automatically.

**Hot/cold temperatures.** A lift that already satisfies the end walls, plus a learnable interior correction that vanishes on them:

$$T=\underbrace{(1-x)}_{\text{linear lift}}+\underbrace{\sin(\pi x)}_{=0\text{ at }x=0,1}\,T_0.$$

At $x=0$: $T=1$. At $x=1$: $T=0$. Exactly, always. Only the insulated top/bottom walls remain for the loss.

In [ ]:
def fields(x, y):
    "Turn raw network outputs into constrained (psi, T)."
    psi0, T0 = model(x, y)
    b = 16 * x * (1 - x) * y * (1 - y)
    psi = b**2 * psi0                       # hard no-slip: psi = 0 and grad(psi) = 0 on all walls
    T = (1 - x) + torch.sin(PI * x) * T0    # hard hot/cold: T = 1 at x=0, T = 0 at x=1
    return psi, T

In [ ]:
def _grad(o, i): 
    return torch.autograd.grad(o, i, torch.ones_like(o), create_graph=True)[0]

yb = torch.linspace(0, 1, 20, device=device).reshape(-1, 1)
_, T0w = fields(torch.zeros_like(yb), yb)
_, T1w = fields(torch.ones_like(yb),  yb)
print(f"T at x=0 (want 1): {T0w.min().item():.4f}..{T0w.max().item():.4f}")
print(f"T at x=1 (want 0): {T1w.min().item():.4f}..{T1w.max().item():.4f}")
xc = torch.zeros(30, 1, device=device, requires_grad=True)
yc = torch.rand(30, 1, device=device, requires_grad=True)
psi, _ = fields(xc, yc)
print(f"no-slip at x=0: max|u|={_grad(psi,yc).abs().max().item():.2e}  max|v|={_grad(psi,xc).abs().max().item():.2e}")

## 8 — The residuals: vorticity transport + energy

Two residuals. From $\psi$ we build velocity ($u=\psi_y$, $v=-\psi_x$) and vorticity ($\zeta=-\nabla^2\psi$); then:

$$
\begin{aligned}
f_1 &= u\,\zeta_x + v\,\zeta_y - Pr\,\nabla^2\zeta - Ra\,Pr\,T_x \quad&&(\text{vorticity transport})\\
f_3 &= u\,T_x + v\,T_y - \nabla^2 T \quad&&(\text{energy})
\end{aligned}
$$

The $\psi$–$\zeta$ relation $\nabla^2\psi+\zeta=0$ is satisfied *by definition* (we set $\zeta=-\nabla^2\psi$), so it needs no residual. Note $\nabla^2\zeta$ makes this a **fourth-order** derivative of $\psi$.

In [ ]:
def grad(o, i): 
    return torch.autograd.grad(o, i, torch.ones_like(o), create_graph=True)[0]

def residuals(x, y):
    psi, T = fields(x, y)
    psi_x, psi_y = grad(psi, x), grad(psi, y)
    u, v = psi_y, -psi_x                          # continuity exact by construction
    psi_xx, psi_yy = grad(psi_x, x), grad(psi_y, y)
    zeta = -(psi_xx + psi_yy)                     # vorticity
    zeta_x, zeta_y = grad(zeta, x), grad(zeta, y)
    lap_zeta = grad(zeta_x, x) + grad(zeta_y, y)  # 4th-order in psi
    T_x, T_y = grad(T, x), grad(T, y)
    lap_T = grad(T_x, x) + grad(T_y, y)
    f1 = u*zeta_x + v*zeta_y - PR*lap_zeta - RA_PR*T_x   # vorticity transport
    f3 = u*T_x + v*T_y - lap_T                           # energy
    return f1, f3

## 9 — The Composite loss

The vorticity residual is divided by $Ra$ before squaring (Fowler et al.) to stop the large buoyancy scale from dominating — the vorticity-formulation analog of the momentum scaling we discussed. The only boundary term is the insulated top/bottom ($\partial T/\partial y=0$).

In [ ]:
def compute_losses():
    f1, f3 = residuals(x_f, y_f)
    loss_pde = ((f1 / RA)**2).mean() + (f3**2).mean()   # vorticity residual scaled by Ra
    _, T_b = fields(x_bot, y_bot); ty_b = grad(T_b, y_bot)
    _, T_t = fields(x_top, y_top); ty_t = grad(T_t, y_top)
    loss_bc = (ty_b**2).mean() + (ty_t**2).mean()       # insulated walls (soft)
    return loss_pde, loss_bc

def total_loss():
    lp, lb = compute_losses()
    return lp + lb

lp, lb = compute_losses()
print(f"untrained:  loss_pde={lp.item():.3e}  loss_bc={lb.item():.3e}")

## 10 — Two-phase training: Adam → L-BFGS

A short Adam warm-up, then L-BFGS (the workhorse for this problem — Fowler et al. rely on it). We log the loss and the three benchmark diagnostics so you can watch convection **develop**: `u_max` and `v_max` climb from ~0, and `Nu` lifts off 1 toward 2.243 as the flow strengthens.

**This is a heavy solve** — 4th-order autograd, and convection takes many L-BFGS iterations to appear. On CPU it crawls; on a GPU it's far more comfortable. Increase the L-BFGS `max_iter` / rounds until the diagnostics plateau near the benchmark. If the flow is slow to start, that's expected — it develops gradually, not suddenly.

In [ ]:
history = {"epoch": [], "pde": [], "bc": [], "Nu": [], "u_max": [], "v_max": []}

def record(epoch):
    lp, lb = compute_losses(); um, vm = peak_velocities()
    history["epoch"].append(epoch); history["pde"].append(lp.item()); history["bc"].append(lb.item())
    history["Nu"].append(nusselt_hot_wall()); history["u_max"].append(um); history["v_max"].append(vm)
    print(f"[{epoch:5d}]  pde={lp.item():.3e} bc={lb.item():.3e}  "
          f"Nu={history['Nu'][-1]:.3f}  u_max={um:.3f}  v_max={vm:.3f}")

# --- Adam warm-up ---
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
ADAM_ITERS = 500
record(0)
for epoch in range(1, ADAM_ITERS + 1):
    opt.zero_grad(); total_loss().backward(); opt.step()
    if epoch % 100 == 0: record(epoch)

# --- L-BFGS refinement (run several rounds; increase until diagnostics plateau) ---
LBFGS_ROUNDS, LBFGS_ITERS = 20, 200
step = ADAM_ITERS
for r in range(LBFGS_ROUNDS):
    opt = torch.optim.LBFGS(model.parameters(), lr=1.0, max_iter=LBFGS_ITERS, max_eval=int(LBFGS_ITERS*1.3),
                            tolerance_grad=1e-12, tolerance_change=1e-14,
                            history_size=50, line_search_fn='strong_wolfe')
    def closure():
        opt.zero_grad(); L = total_loss(); L.backward(); return L
    opt.step(closure); step += LBFGS_ITERS
    record(step)
print("done — compare the final Nu / u_max / v_max against BENCHMARK[RA]")

## 11 — The Loss History, Both Phases

In [ ]:
# convergence history: watch convection develop
epoch = np.array(history["epoch"])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].semilogy(epoch, history["pde"], 'o-', color='#1A1A2E', ms=3, label='PDE residual')
ax[0].semilogy(epoch, np.maximum(history["bc"],1e-14), ':', color='#6D4C9E', label='insulated BC')
ax[0].set_xlabel('iteration'); ax[0].set_ylabel('loss'); ax[0].set_title('Loss history'); ax[0].legend(fontsize=8)
ax2 = ax[1]; ax3 = ax2.twinx()
ax2.plot(epoch, history["Nu"], 'o-', color='#2A9D8F', ms=3, label='Nu')
ax2.axhline(BENCHMARK[RA]['Nu'], ls=':', color='#2A9D8F', lw=1)
ax3.plot(epoch, history["u_max"], '^-', color='#E76F51', ms=3, alpha=0.8, label='u_max')
ax3.plot(epoch, history["v_max"], 's-', color='#3B6FB0', ms=3, alpha=0.8, label='v_max')
ax3.axhline(BENCHMARK[RA]['v_max'], ls=':', color='#3B6FB0', lw=1)
ax2.set_xlabel('iteration'); ax2.set_ylabel('Nu', color='#2A9D8F'); ax3.set_ylabel('peak velocity')
ax2.set_title('Convection developing toward the benchmark')
h1,l1=ax2.get_legend_handles_labels(); h2,l2=ax3.get_legend_handles_labels()
ax2.legend(h1+h2, l1+l2, fontsize=8, loc='center right')
plt.tight_layout(); plt.show()

## 12 — Results and validation

Compare the trained diagnostics against the benchmark,then plot the fields.

In [ ]:
um, vm = peak_velocities()
b = BENCHMARK[RA]
print(f"quantity     PINN        de Vahl Davis   rel. error")
print(f"Nu          {nusselt_hot_wall():8.3f}    {b['Nu']:8.3f}      {abs(nusselt_hot_wall()-b['Nu'])/b['Nu']*100:5.1f}%")
print(f"u_max       {um:8.3f}    {b['u_max']:8.3f}      {abs(um-b['u_max'])/b['u_max']*100:5.1f}%")
print(f"v_max       {vm:8.3f}    {b['v_max']:8.3f}      {abs(vm-b['v_max'])/b['v_max']*100:5.1f}%")

In [ ]:
# field contours: temperature, stream function, velocity magnitude
n = 200
gx = torch.linspace(0, 1, n, device=device)
X, Y = torch.meshgrid(gx, gx, indexing='ij')
xf = X.reshape(-1,1).requires_grad_(True); yf = Y.reshape(-1,1).requires_grad_(True)
psi, T = fields(xf, yf)
u = grad(psi, yf); v = -grad(psi, xf)
speed = torch.sqrt(u**2 + v**2)
def G(t): return t.detach().reshape(n, n).cpu().numpy()
Xn, Yn = X.cpu().numpy(), Y.cpu().numpy()

fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
for a, (D, ttl, cm) in zip(ax, [(G(T),'Temperature $T$','inferno'),
                                (G(psi),'Stream function $\\psi$','viridis'),
                                (G(speed),'Speed $|\\mathbf{u}|$','magma')]):
    cf = a.contourf(Xn, Yn, D, levels=40, cmap=cm); fig.colorbar(cf, ax=a, fraction=0.046)
    a.set_title(ttl); a.set_xlabel('x'); a.set_ylabel('y'); a.set_aspect('equal')
plt.suptitle(f'Natural convection, Ra = {RA:.0e}', y=1.02)
plt.tight_layout(); plt.show()

- **Why this converges and the primitive form doesn't.** Continuity is exact, pressure is gone, and no-slip + wall temperatures are hard-constrained — so the optimizer only has to find the interior circulation, not fight a huge pressure or discover flow from a dead-flat start. The conduction trap that stops $(u,v,p,T)$ is sidestepped by construction.
- **It develops gradually.** Convection appears as a slow, monotonic climb in `u_max`/`v_max` over many L-BFGS iterations, with `Nu` lifting off 1 only once the flow is strong enough to distort the temperature field. If your diagnostics are short of the benchmark, **train longer** (more L-BFGS rounds) before concluding anything.
- **Higher $Ra$ is harder.** At $Ra=10^5$–$10^6$ the boundary layers thin and Fowler et al. report the optimizer briefly converging to a *sign-flipped* wrong solution before finding the right one. Expect longer training and add interior points.
- **Reference.** Formulation and PINN benchmark: Fowler, McDevitt & Roy, *Sci. Rep.* **14**, 15203 (2024). Underlying CFD benchmark: de Vahl Davis, *Int. J. Numer. Methods Fluids* **3**, 249–264 (1983).


## 13 — Exercises


1. **Plane strain instead of plane stress.** Swap the Hooke's-law constants for the plane-strain forms (thick components rather than thin plates) and re-run. Confirm the *stress* field is essentially unchanged — recall from Section 2 that the Kirsch stresses carry no $E$ or $\nu$. This is a check you can predict the answer to, which is the point.
2. **Move the outer wall.** Change `L` from 4 to 2, then to 8. Watch how the far-field boundary's distance from the hole affects the recovered SCF, and how the collocation cost scales.
3. **Finite-width drift.** Replace the Kirsch tractions on the outer edges with *plain uniform tension* $\sigma_0$. The hole no longer sits infinitely far from a boundary, so the exact $K_t=3$ no longer holds — it creeps upward. Measure by how much for a few values of `L/a`. This is the real-specimen version of the problem, and the answer is a finite-width correction you can look up and compare against.
4. **Close the SCF gap.** Double `N_near`, then raise the hole-traction weight, and plot the final SCF against each. You'll watch the smoothing floor lift — and find out how much it costs to buy the last few percent toward 3.
5. **Validate the displacements, not just the stresses.** We validated on stress, where the factor of 3 lives, and left Kirsch's displacement field uncoded. Add it, and check the network's raw $(u, v)$ outputs directly. (Remember: displacements *do* depend on $E$ and $\nu$.)
6. **Elliptical hole.** Replace the circular hole with an ellipse of semi-axes $a, b$. The stress-concentration factor becomes $K_t = 1 + 2a/b$ — a new analytical target, reached with exactly the same machinery. A circle ($a=b$) recovers 3.

## 14 — Failure modes to watch for

Everything in this table is something that either bit this notebook during development or is one keystroke away from doing so. The bolded row is the one unique to this problem — and the most useful debugging instinct it teaches.

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| `too many values to unpack (expected 2)` | leftover `u, v, p` triple copied from the Navier–Stokes cavity — elasticity has no pressure | final layer → **2** outputs; `forward` returns `(u, v)`; then **re-instantiate** `model` (redefining the class isn't enough in Jupyter) |
| **boundary loss tiny, but SCF still ≈ 0** | the network fits the edges while interior equilibrium lags — the concentration lives entirely in the residual | **keep training.** Don't stop when `bc` is small; the SCF only rises as `pde` falls two more orders of magnitude |
| SCF plateaus below 3 (≈2.9) | the network smooths the single sharpest point in the domain | more near-hole points; raise the hole-traction weight; a wider net. A residual "smoothing floor" remains — name it, don't hide it |
| interior points appear *inside* the hole | rejection mask inverted or missing | assert `min r ≥ a` right after sampling |
| autograd error when computing boundary stresses | `requires_grad_(True)` called **before** `.to(device)` → the tensor is no longer a leaf | move to device **first**, then set grad (the `edge()` helper does this) |
| stresses off by a constant factor | the shear law mis-coded | it's $\sigma_{xy}=C(1-\nu)\,\varepsilon_{xy}$, since $E/(1+\nu)=C(1-\nu)$ — verified symbolically in Section 8 |
| `NaN` at the crown | `atan2(0, 0)` is undefined at the origin corner | evaluate the crown at `x = 1e-6`, not exactly 0 |
| loss stalls early in L-BFGS | line search stuck in a flat region | the round-based restart (fresh optimizer each round) resets the history and frees it |

## 15 — Vocabulary

Only the terms this tutorial adds to the series:

- **Airy stress function** — a single scalar $\phi$ whose second derivatives are the stresses; it satisfies equilibrium automatically, collapsing three unknowns into one.
- **Biharmonic equation** — $\nabla^4\phi = 0$, the compatibility condition the Airy function must obey; a fourth-order PDE.
- **Compatibility** — the requirement that strains derive from a single-valued displacement field; what forces $\phi$ to be biharmonic.
- **Plane stress vs. plane strain** — thin-plate ($\sigma_z=0$) vs. thick-body ($\varepsilon_z=0$) 2D idealizations; they differ only in the elastic constants.
- **Traction vector** — $\mathbf{t}=\boldsymbol\sigma\cdot\mathbf{n}$, the force per area on a surface with normal $\mathbf{n}$; zero on a free surface, prescribed on a loaded one.
- **Stress-concentration factor ($K_t$)** — the ratio of peak local stress to the applied stress; 3 for a small circular hole in uniaxial tension.
- **Hoop stress ($\sigma_{\theta\theta}$)** — the circumferential stress; on the hole rim it carries the entire concentration.
- **Symmetry boundary condition** — using a plane of symmetry to model a fraction of the domain; the normal displacement and the shear stress both vanish on it.
- **Kirsch solution** — the 1898 closed-form stress field for an infinite plate with a circular hole under remote tension; our reference.

## 16 — What You Have Built (and Where the Series Goes)

Tutorial 2 stripped the problem down to one scalar equation. This one climbed partway back up — and every step of the climb was a new idea. The network output became a **vector** $(u,v)$. The quantity we cared about, stress, was never predicted but **differentiated out** of the solution. The domain stopped being a box and grew a **curved internal boundary**. And the target wasn't a smooth field but a single number every mechanical engineer knows: the stress-concentration factor of **3**. You recovered it to **2.93 — within 2.3%** — with the full stress field inside **0.83%**.

The physics was richer, so the story lived in the chain, not the edges:

| Stage | What the network does | The mechanism |
|---|---|---|
| output | predicts displacements $(u,v)$ | two-column final layer |
| kinematics | strains from displacements | first autograd pass |
| constitution | stresses from strains | Hooke's law (no derivatives) |
| equilibrium | the residual to zero | second autograd pass |

That chain is the whole lesson. In Tutorial 2 the network output *was* the quantity in the PDE; here it sits three steps upstream, and autograd walks the distance. It's the same compositionality the series keeps returning to — Tutorial 1 changed the *residual*, Tutorial 2 changed the *boundary term*, and here the *unknown itself* moves upstream of the equation — but the recipe never changes.

And you met the honest limit again, sharper this time. The SCF didn't quite reach 3: the network **smooths the single steepest point in the domain**, the crown of the hole, and gives up that last 2%. It's Tutorial 2's troublesome corners intensified — where the gradient is fiercest, a smoothness-biased network rounds the peak. You watched it happen live, too: the SCF stayed near zero through all of Adam while the boundaries fit perfectly, and only climbed once L-BFGS drove the interior residual down. **The concentration lives in the residual, and it emerges last.**

**The honest verdict on this problem:** a well-resolved finite-element mesh nails $K_t=3$ essentially exactly, and faster. This isn't the problem where PINNs win. But consider what the network *did* deliver — the entire stress field of a holed plate, right to under a percent, from one mesh-free model of 12,802 parameters with no elements, no mesh generation, and a curved boundary handled by simply sampling points on it. That last part is the tell: the mesh-free handling of awkward geometry is exactly the edge the series keeps climbing toward.

---

### Get the Notebook

The full series — every tutorial, with all plots and training logs — lives on GitHub:

**[github.com/rraj-io/PINN_for_Mechanical_Engineer](https://github.com/rraj-io/PINN_for_Mechanical_Engineer)**

```bash
git clone https://github.com/rraj-io/PINN_for_Mechanical_Engineer.git
cd PINN_for_Mechanical_Engineer
```

New articles and shorter notes on PINNs, neural operators, CFD, and scientific machine learning publish regularly on Substack: **[rajrohit403.substack.com](https://rajrohit403.substack.com/)**

*Next up: two fields that finally talk to each other — buoyancy-driven flow, where temperature and velocity couple, and the honest question of how far up the Rayleigh number a PINN can climb. — Rohit Raj · HalfQubiX*